# Interactive generative zoom
Select **Runtime → Change runtime type → GPU** (a T4 is a good starting point), then **Runtime → Run all**. The notebook contains the demo source and browser UI. Model weights download from Hugging Face on first use.

In the embedded demo, upload a photo and click a detail with **Wait for Qwen** selected under **Prompt mode**. Leave the optional prompt hint blank to start from the image alone. Choose 2× or 4× under **Zoom per click**. Each click smoothly zooms into the selected region and reveals the refined image. Click the result to explore another detail. Qwen identifies a visible feature and gives the image model one grounded refinement to make. In **Backend**, choose **SD-Turbo** on a T4. For **Z-Image-Turbo · 1024×1024**, choose an L4, A100, or another GPU with native BF16 support and enough VRAM. The selected image model downloads on first use. Keep this runtime connected while exploring.

In [ ]:
#@title Unpack the bundled demo files
from pathlib import Path
from tempfile import mkdtemp
project = Path(mkdtemp(prefix='generative-zoom-'))
files = {
    'app.py': '"""Local zoom server; the browser supplies bounded refinement context per request."""\nimport argparse\nimport base64\nimport io\nimport threading\nimport time\nfrom importlib.util import find_spec\nfrom pathlib import Path\n\nimport json\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\nfrom PIL import Image\nfrom pydantic import BaseModel, Field, ValidationError, model_validator\nfrom typing import Literal, Tuple\n\nfrom director import DirectorRequest, PromptDirector\nfrom generator import Generator\nfrom imaging import FRAME_SIZE, crop_focus, decode_zoom_crop, feather_patch, preserve_source_detail\n\nROOT = Path(__file__).resolve().parent\ngenerator = Generator()\nmodel_lock = threading.Lock()\nollama_lock = threading.Lock()\nprompt_director = PromptDirector()\ndefault_backend = \'diffusion\'\nwarmup_status = {\'state\': \'idle\', \'message\': \'\'}\nSTATIC_FILES = {\n    \'/\': (\'index.html\', \'text/html; charset=utf-8\'),\n    \'/static/app.js\': (\'app.js\', \'text/javascript; charset=utf-8\'),\n    \'/static/prompting.mjs\': (\'prompting.mjs\', \'text/javascript; charset=utf-8\'),\n    \'/static/style.css\': (\'style.css\', \'text/css; charset=utf-8\'),\n}\n\n\nclass ModelBusyError(RuntimeError):\n    pass\n\n\ndef acquire_model():\n    if warmup_status[\'state\'] == \'warming\' or not model_lock.acquire(blocking=False):\n        raise ModelBusyError(\'The model is busy. You can use Lanczos while it finishes.\')\n\n\ndef start_warmup():\n    """Prepare the chosen models without blocking the page or the Lanczos path."""\n    global warmup_status\n    warmup_status = {\'state\': \'warming\', \'message\': \'Preparing the image model…\'}\n\n    def prepare():\n        global warmup_status\n        try:\n            with model_lock:\n                generator.warmup(default_backend)\n        except Exception as exc:\n            print(f\'Model warm-up failed: {exc}\', flush=True)\n            warmup_status = {\'state\': \'failed\', \'message\': str(exc)}\n        else:\n            warmup_status = {\'state\': \'ready\', \'message\': \'Image model ready.\'}\n\n    worker = threading.Thread(target=prepare, daemon=True, name=\'model-warmup\')\n    worker.start()\n    return worker\n\n\ndef direct(req: DirectorRequest):\n    # Ollama runs in a separate process. Colab\'s Qwen and diffusion share one GPU.\n    lock = ollama_lock if prompt_director.backend == \'ollama\' else model_lock\n    if lock is model_lock:\n        acquire_model()\n    elif not lock.acquire(blocking=False):\n        raise ModelBusyError(\'Ollama is already reading a crop. Try again shortly.\')\n    try:\n        return prompt_director.next(req)\n    finally:\n        prompt_director.stage = None\n        lock.release()\n\n\nclass Request(BaseModel):\n    image: str = Field(max_length=8_000_000)\n    center: Tuple[float, float]\n    zoom_factor: Literal[2, 4] = 2\n    prompt: str = Field(default=\'\', max_length=2000)\n    backend: Literal[\'upscale\', \'diffusion\', \'z-image\', \'openvino\'] = \'diffusion\'\n    refinement_steps: int = Field(default=1, ge=1, le=8)\n    seed: int = Field(default=42, ge=0, le=2147483647)\n    sharpen: float = Field(default=0, ge=0, le=3)\n    feather: int = Field(default=32, ge=0, le=128)\n    preserve_structure: bool = True\n\n    @model_validator(mode=\'after\')\n    def validate_settings(self):\n        if not all(0 <= v <= 1 for v in self.center):\n            raise ValueError(\'Center must lie inside the image\')\n        if self.backend in (\'diffusion\', \'openvino\') and self.refinement_steps > 4:\n            raise ValueError("SD-Turbo supports up to four refinement steps")\n        return self\n\ndef encode(image, compress_level=6):\n    buffer = io.BytesIO()\n    image.save(buffer, format=\'PNG\', compress_level=compress_level)\n    return \'data:image/png;base64,\' + base64.b64encode(buffer.getvalue()).decode()\n\ndef zoom(req: Request):\n    start = time.perf_counter()\n    crop, bounds = decode_zoom_crop(req.image, req.center, req.zoom_factor)\n    output_size = (FRAME_SIZE, FRAME_SIZE)\n    if req.backend == \'upscale\':\n        view = encode(crop.resize(output_size, Image.Resampling.LANCZOS))\n        return {\'image\': view, \'baseline\': view, \'raw\': view, \'crop\': bounds,\n                \'seconds\': time.perf_counter() - start, \'model\': \'Lanczos baseline\',\n                \'load_seconds\': 0, \'inference_seconds\': 0, \'denoising_steps\': 0, \'start_timestep\': None}\n\n    acquire_model()\n    try:\n        try:\n            raw, timing = generator.generate(crop, req.prompt, req.refinement_steps, req.seed, req.backend, req.sharpen)\n        except Exception as exc:\n            raise RuntimeError(f\'Generation failed: {exc}\') from exc\n    finally:\n        model_lock.release()\n\n    baseline_view = crop.resize(output_size, Image.Resampling.LANCZOS)\n    raw_view = raw.resize(output_size, Image.Resampling.LANCZOS)\n    edited_view = raw_view\n    if req.preserve_structure:\n        edited_view = preserve_source_detail(baseline_view, raw_view, crop_focus(req.center, bounds))\n    blended = feather_patch(baseline_view, edited_view, 2 * req.feather)\n    compression = 1 if req.backend == \'openvino\' else 6\n    return {\'image\': encode(blended, compression), \'baseline\': encode(baseline_view, compression),\n            \'raw\': encode(raw_view, compression), \'crop\': bounds,\n            \'seconds\': time.perf_counter() - start, **timing}\n\n\nclass Handler(BaseHTTPRequestHandler):\n    def log_request(self, code=\'-\', size=\'-\'):\n        polling = self.path.partition(\'?\')[0] in (\'/api/config\', \'/api/director/status\')\n        if not (polling and code == 200):\n            super().log_request(code, size)\n\n    def respond(self, status, body, content_type):\n        try:\n            self.send_response(status)\n            self.send_header(\'Content-Type\', content_type)\n            self.send_header(\'Content-Length\', str(len(body)))\n            if content_type == \'application/json\':\n                self.send_header(\'Cache-Control\', \'no-store\')\n            self.end_headers()\n            self.wfile.write(body)\n        except (BrokenPipeError, ConnectionResetError, ConnectionAbortedError):\n            # A browser can leave a request while the model is still computing.\n            self.close_connection = True\n\n    def do_GET(self):\n        path = self.path.partition(\'?\')[0]\n        if path == \'/api/config\':\n            config = {\'realtime\': default_backend == \'openvino\',\n                      \'openvino_available\': find_spec(\'openvino_genai\') is not None,\n                      \'background_prompting\': prompt_director.backend == \'ollama\',\n                      \'warmup\': warmup_status}\n            self.respond(200, json.dumps(config).encode(), \'application/json\')\n            return\n        if path == \'/api/director/status\':\n            self.respond(200, json.dumps({\'stage\': prompt_director.stage}).encode(), \'application/json\')\n            return\n        if path == \'/favicon.ico\':\n            self.respond(204, b\'\', \'image/x-icon\')\n            return\n        if path not in STATIC_FILES:\n            self.respond(404, b\'Not found\', \'text/plain\')\n            return\n        name, mime = STATIC_FILES[path]\n        self.respond(200, (ROOT/\'static\'/name).read_bytes(), mime)\n\n    def do_POST(self):\n        if self.path not in (\'/api/zoom\', \'/api/director\'):\n            self.respond(404, b\'Not found\', \'text/plain\')\n            return\n        try:\n            length = int(self.headers.get(\'Content-Length\', \'0\'))\n            if not 0 < length <= 8_100_000:\n                raise ValueError(\'Invalid request size\')\n            body = self.rfile.read(length)\n            if self.path == \'/api/director\':\n                req = DirectorRequest.model_validate_json(body)\n                output = direct(req)\n            else:\n                req = Request.model_validate_json(body)\n                output = zoom(req)\n            self.respond(200, json.dumps(output).encode(), \'application/json\')\n        except ModelBusyError as exc:\n            self.respond(409, json.dumps({\'detail\': str(exc)}).encode(), \'application/json\')\n        except ValidationError as exc:\n            error = exc.errors(include_url=False, include_input=False)[0]\n            field = \' \'.join(str(part).replace(\'_\', \' \') for part in error[\'loc\'])\n            message = error[\'msg\'].removeprefix(\'Value error, \')\n            detail = f\'{field.capitalize()} — {message}\' if field else message\n            self.respond(400, json.dumps({\'detail\': detail}).encode(), \'application/json\')\n        except Exception as exc:\n            print(f\'{self.path} failed: {exc}\', flush=True)\n            status = 400 if isinstance(exc, ValueError) else 500\n            self.respond(status, json.dumps({\'detail\': str(exc)}).encode(), \'application/json\')\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=\'Run the generative zoom server.\')\n    parser.add_argument(\'--warmup\', action=\'store_true\',\n                        help=\'Prepare the selected pipeline in the background on startup.\')\n    parser.add_argument(\'--realtime\', action=\'store_true\',\n                        help=\'Prefer the local OpenVINO backend in the browser.\')\n    args = parser.parse_args()\n    if args.realtime:\n        default_backend = \'openvino\'\n    with ThreadingHTTPServer((\'127.0.0.1\', 8000), Handler) as server:\n        if args.warmup:\n            start_warmup()\n        print(\'Open http://127.0.0.1:8000\', flush=True)\n        server.serve_forever()\n',
    'director.py': '"""Crop-grounded detail directions from Qwen."""\nimport base64\nimport io\nimport json\nimport os\nimport secrets\nimport time\nimport urllib.error\nimport urllib.request\nfrom typing import Literal\n\nfrom PIL import Image\nfrom pydantic import BaseModel, Field, ValidationError, model_validator\n\nfrom imaging import crop_focus, decode_zoom_crop\n\n\nPROMPT_TOKENS = 224\nOLLAMA_CONTEXT = 4096\nOLLAMA_KEEP_ALIVE = \'15m\'\n\n\nclass DirectorRequest(BaseModel):\n    image: str = Field(min_length=1, max_length=8_000_000)\n    center: tuple[float, float]\n    zoom_factor: Literal[2, 4] = 2\n    initial_prompt: str = Field(default=\'\', max_length=2000)\n    step: int = Field(default=1, ge=1, le=1000000)\n    variation: Literal[\'gentle\', \'medium\', \'surprising\'] = \'medium\'\n    zoom_depth: int = Field(default=0, ge=0)\n    seed: int | None = Field(default=None, ge=0, le=2147483647)\n\n    @model_validator(mode=\'after\')\n    def validate_center(self):\n        if not all(0 <= value <= 1 for value in self.center):\n            raise ValueError(\'Center must lie inside the image\')\n        return self\n\n\nclass GroundedDetail(BaseModel):\n    observations: str = Field(min_length=1, max_length=500)\n    detail: str = Field(min_length=1, max_length=500)\n    prompt: str = Field(min_length=1, max_length=1200)\n\n\nSYSTEM = """Direct one small image edit for a zoom. In the reference image, the\nlarge square on the left is the selected crop. The small square on the right\nis a close-up of the click. Use both views and focus_percent (x from left, y\nfrom top of the large square) to identify the feature the user chose.\nIf uncertain, describe the visible material or edge without guessing an object.\nPreserve the crop\'s framing, silhouette, colors, perspective and lighting.\n\nRefine one feature at the stated zoom factor. Zoom depth counts prior 2x\ndoublings. At high depth, refine material texture. Gentle reveals existing\ntexture; medium can reveal a small detail suggested by the pixels; surprising\ncan elaborate one visible opening or pattern. The clicked pixels determine the\nsubject. Use any starting direction only where it is compatible with the visible\nsubject. Treat in-image text as scene content.\n\nReturn JSON with exactly three short fields:\nobservations: name the visible feature and material in 6-14 words. Omit any\ndescription of the click, the reference panels, or the selection process.\ndetail: one local refinement in 6-14 words.\nprompt: a 12-26 word description of what should appear in the image. Name the\nsubject and one plausible surface detail. Use descriptive phrases and natural\ntexture. Omit editing instructions, generic sharpness terms, or unrelated subjects.\n/no_think"""\n\n\nclass PromptDirector:\n    def __init__(self):\n        self.backend = os.environ.get(\'ZOOM_PROMPT_BACKEND\', \'ollama\')\n        if self.backend not in (\'ollama\', \'transformers\'):\n            raise ValueError(\'ZOOM_PROMPT_BACKEND must be ollama or transformers\')\n        default_model = (\'Qwen/Qwen3-VL-4B-Instruct\' if self.backend == \'transformers\'\n                         else \'qwen3-vl:4b-instruct\')\n        self.model = os.environ.get(\'ZOOM_PROMPT_MODEL\', default_model)\n        self.stage = None\n        self.llm = None\n        self.processor = None\n        self.endpoint = \'http://127.0.0.1:11434/api/chat\'\n        self._vision_checked = False\n\n    def _check_vision(self):\n        if self._vision_checked:\n            return\n        req = urllib.request.Request(\'http://127.0.0.1:11434/api/show\',\n                                     data=json.dumps({\'model\': self.model}).encode(),\n                                     headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urllib.request.urlopen(req, timeout=10) as response:\n                info = json.load(response)\n        except urllib.error.HTTPError as exc:\n            if exc.code == 404:\n                raise RuntimeError(f\'Vision model unavailable. Run: ollama pull {self.model}\') from exc\n            raise RuntimeError(\'Cannot inspect the local vision model. Check Ollama.\') from exc\n        except (urllib.error.URLError, TimeoutError, OSError, ValueError) as exc:\n            raise RuntimeError(\'Cannot reach the local vision model. Start Ollama with: ollama serve\') from exc\n        if not isinstance(info, dict) or \'vision\' not in info.get(\'capabilities\', []):\n            raise RuntimeError(\'The configured prompt model cannot read images. Use qwen3-vl:4b-instruct or another vision-capable Ollama model.\')\n        self._vision_checked = True\n\n    def next(self, request: DirectorRequest, _retry=False):\n        start = time.perf_counter()\n        crop, bounds = decode_zoom_crop(request.image, request.center, request.zoom_factor)\n        focus = crop_focus(request.center, bounds)\n        # Put both views in one image so the local model only encodes one attachment.\n        size = crop.width * 2 // 5\n        left, top = (\n            max(0, min(crop.width - size, round(value * crop.width) - size // 2))\n            for value in focus\n        )\n        closeup = crop.crop((left, top, left + size, top + size))\n        closeup = closeup.resize((128, 128), Image.Resampling.LANCZOS)\n        crop.thumbnail((320, 320), Image.Resampling.LANCZOS)\n        reference = Image.new(\'RGB\', (crop.width + 8 + closeup.width, crop.height), \'#181818\')\n        reference.paste(crop, (0, 0))\n        reference.paste(closeup, (crop.width + 8, (crop.height - closeup.height) // 2))\n        seed = request.seed if request.seed is not None else secrets.randbelow(2147483647)\n        context = {\n            \'variation\': request.variation,\n            \'focus_percent\': [round(100 * value, 1) for value in focus],\n            \'zoom_factor\': request.zoom_factor,\n            \'zoom_depth\': request.zoom_depth,\n        }\n        if hint := request.initial_prompt.strip():\n            context[\'starting_direction\'] = hint\n        if _retry:\n            context[\'revision\'] = \'Return valid JSON. Refine a specific feature of the same visible object.\'\n        user = json.dumps(context)\n        temperature = {\'gentle\': 0.2, \'medium\': 0.4, \'surprising\': 0.65}[request.variation]\n        generate = self._transformers if self.backend == \'transformers\' else self._ollama\n        content = generate(reference, user, temperature, seed)\n        try:\n            beat = GroundedDetail.model_validate_json(content)\n            beat = GroundedDetail(observations=beat.observations.strip(),\n                                prompt=beat.prompt.strip(), detail=beat.detail.strip())\n        except ValidationError as exc:\n            if self.backend == \'transformers\' and not _retry:\n                revised = self.next(request, _retry=True)\n                revised[\'seconds\'] = time.perf_counter() - start\n                return revised\n            raise RuntimeError(\'The prompt model returned an incomplete detail direction. Try the step again.\') from exc\n        return {**beat.model_dump(), \'model\': self.model, \'step\': request.step,\n                \'seed\': seed, \'seconds\': time.perf_counter() - start}\n\n    def report_stage(self, stage):\n        self.stage = stage\n        print(f\'[Qwen] {stage}\', flush=True)\n\n    def _transformers(self, reference: Image.Image, user, temperature, seed):\n        self.report_stage(\'Preparing the Qwen vision runtime\')\n        import torch\n        from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration\n\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'Select a GPU runtime in Colab and run the notebook again.\')\n        if self.llm is None:\n            self.report_stage(\'Loading the Qwen vision processor\')\n            self.processor = AutoProcessor.from_pretrained(self.model)\n            quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                               bnb_4bit_compute_dtype=torch.float16)\n            self.report_stage(\'Loading 4-bit Qwen vision weights (downloads on first use)\')\n            self.llm = Qwen3VLForConditionalGeneration.from_pretrained(\n                self.model, device_map={\'\': 0}, dtype=torch.float16,\n                quantization_config=quantization).eval()\n        self.report_stage(\'Reading the selected crop and writing a detail direction\')\n        messages = [\n            {\'role\': \'system\', \'content\': [{\'type\': \'text\', \'text\': SYSTEM}]},\n            {\'role\': \'user\', \'content\': [\n                {\'type\': \'image\', \'image\': reference},\n                {\'type\': \'text\', \'text\': user},\n            ]},\n        ]\n        inputs = self.processor.apply_chat_template(\n            messages, tokenize=True, add_generation_prompt=True,\n            return_dict=True, return_tensors=\'pt\').to(self.llm.device)\n        with torch.inference_mode(), torch.random.fork_rng(devices=[0]):\n            torch.cuda.manual_seed(seed)\n            ids = self.llm.generate(**inputs, do_sample=True, temperature=temperature,\n                                    top_p=0.9, repetition_penalty=1.1, max_new_tokens=PROMPT_TOKENS)\n        completion = ids[0, inputs.input_ids.shape[1]:]\n        return self.processor.decode(completion, skip_special_tokens=True).strip()\n\n    def _ollama(self, reference: Image.Image, user, temperature, seed):\n        self.report_stage(\'Reading the selected crop with Ollama\')\n        self._check_vision()\n        image_bytes = io.BytesIO()\n        reference.save(image_bytes, format=\'PNG\')\n        encoded = base64.b64encode(image_bytes.getvalue()).decode()\n        payload = {\n            \'model\': self.model,\n            \'messages\': [{\'role\': \'system\', \'content\': SYSTEM},\n                         {\'role\': \'user\', \'content\': user, \'images\': [encoded]}],\n            \'format\': GroundedDetail.model_json_schema(),\n            \'think\': False,\n            \'stream\': False,\n            \'keep_alive\': OLLAMA_KEEP_ALIVE,\n            \'options\': {\'temperature\': temperature,\n                        \'top_p\': 0.9, \'repeat_penalty\': 1.1, \'seed\': seed,\n                        \'num_ctx\': OLLAMA_CONTEXT, \'num_predict\': PROMPT_TOKENS},\n        }\n        result = self._ollama_chat(payload)\n        try:\n            if not isinstance(result, dict) or not result.get(\'done\') or result.get(\'done_reason\') == \'length\':\n                raise ValueError(\'Incomplete generation\')\n            return result[\'message\'][\'content\']\n        except (KeyError, TypeError, ValueError) as exc:\n            raise RuntimeError(\'The prompt model returned an incomplete detail direction. Try again at a tighter zoom.\') from exc\n\n    def _ollama_chat(self, payload):\n        req = urllib.request.Request(self.endpoint, data=json.dumps(payload).encode(),\n                                     headers={\'Content-Type\': \'application/json\'})\n        try:\n            with urllib.request.urlopen(req, timeout=90) as response:\n                return json.load(response)\n        except urllib.error.HTTPError as exc:\n            if exc.code == 404:\n                raise RuntimeError(f\'Prompt model unavailable. Run: ollama pull {self.model}\') from exc\n            raise RuntimeError(\'The local vision model failed. Check that Ollama supports the selected vision model and try again.\') from exc\n        except TimeoutError as exc:\n            raise RuntimeError(\'Ollama did not finish the prompt in time. Try a tighter zoom or a smaller vision model.\') from exc\n        except (urllib.error.URLError, OSError) as exc:\n            raise RuntimeError(\'Cannot reach the local prompt model. Start Ollama with: ollama serve\') from exc\n        except (ValueError, TypeError) as exc:\n            raise RuntimeError(\'Ollama returned an invalid response. Try the step again.\') from exc\n',
    'generator.py': '"""Lazy model loading: the baseline needs neither torch nor model downloads."""\nimport os\nimport time\nfrom pathlib import Path\nfrom imaging import sharpen_input\nfrom PIL import Image\n\nZ_IMAGE = "Tongyi-MAI/Z-Image-Turbo"\nOPENVINO_MODEL = \'rupeshs/sd-turbo-openvino\'\nOPENVINO_ROOT = Path(__file__).resolve().parent / \'data\' / \'openvino\'\nOPENVINO_SCHEDULE = 4\n\n\nclass OpenVinoZoom:\n    """Keep a compiled image-to-image pipeline for the local interactive path."""\n\n    def __init__(self):\n        self.pipe = None\n        self.size = 384\n\n    def warmup(self):\n        if self.pipe is not None:\n            return 0\n        start = time.perf_counter()\n        try:\n            from openvino import Core\n            from openvino_genai import Image2ImagePipeline\n        except ImportError as exc:\n            raise RuntimeError(\'Install the local OpenVINO runtime: pip install -r requirements-openvino.txt\') from exc\n        from huggingface_hub import snapshot_download\n\n        model = OPENVINO_ROOT / \'sd-turbo\'\n        parts = (\'text_encoder\', \'unet\', \'vae_encoder\', \'vae_decoder\')\n        required = [\'model_index.json\', \'scheduler/scheduler_config.json\',\n                    \'tokenizer/tokenizer_config.json\', \'tokenizer/merges.txt\', \'tokenizer/vocab.json\']\n        required.extend(f\'{part}/{name}\' for part in parts\n                        for name in (\'config.json\', \'openvino_model.xml\', \'openvino_model.bin\'))\n        if not all((model / name).is_file() for name in required):\n            print(\'Downloading the OpenVINO SD-Turbo checkpoint to data/openvino...\', flush=True)\n            snapshot_download(OPENVINO_MODEL, revision=\'b5582a9\', local_dir=model,\n                              allow_patterns=[\'*.json\', \'*.xml\', \'*.bin\', \'tokenizer/*.txt\'], max_workers=4)\n\n        tokenizer = model / \'tokenizer\'\n        tokenizer_ready = tokenizer / \'.zoom-tokenizer-i32\'\n        if not tokenizer_ready.is_file():\n            from openvino import Type, save_model\n            from openvino_tokenizers import convert_tokenizer\n            from transformers import CLIPTokenizerFast\n\n            clip = CLIPTokenizerFast.from_pretrained(tokenizer, local_files_only=True)\n            converted = convert_tokenizer(clip, tokenizer_output_type=Type.i32, use_max_padding=True)\n            save_model(converted, tokenizer / \'openvino_tokenizer.xml\')\n            tokenizer_ready.touch()\n\n        preference = os.environ.get(\'ZOOM_OPENVINO_DEVICE\', \'AUTO\').upper()\n        device = preference\n        if device == \'AUTO\':\n            device = \'GPU\' if any(name.split(\'.\')[0] == \'GPU\' for name in Core().available_devices) else \'CPU\'\n\n        def compile_pipeline(target):\n            print(f\'Preparing OpenVINO SD-Turbo ({self.size}px, {target})...\', flush=True)\n            pipeline = Image2ImagePipeline(str(model))\n            pipeline.reshape(1, self.size, self.size, 0.0)\n            pipeline.compile(target, CACHE_DIR=str(OPENVINO_ROOT / \'compiled\'), PERFORMANCE_HINT=\'LATENCY\')\n            return pipeline\n\n        try:\n            self.pipe = compile_pipeline(device)\n        except RuntimeError as exc:\n            if preference != \'AUTO\' or device != \'GPU\':\n                raise\n            print(f\'OpenVINO GPU unavailable, using CPU: {exc}\', flush=True)\n            device = \'CPU\'\n            self.pipe = compile_pipeline(device)\n        self.device = device\n        return time.perf_counter() - start\n\n    def generate(self, image, prompt, steps, seed, sharpen):\n        load_time = self.warmup()\n        import numpy as np\n        from openvino import Tensor\n\n        start = time.perf_counter()\n        image = image.resize((self.size, self.size), Image.Resampling.LANCZOS)\n        image = sharpen_input(image, sharpen)\n        tensor = Tensor(np.asarray(image)[None, ...].copy())\n        result = self.pipe.generate(prompt, tensor, height=self.size, width=self.size,\n                                    num_inference_steps=OPENVINO_SCHEDULE, strength=steps / OPENVINO_SCHEDULE,\n                                    guidance_scale=0.0, rng_seed=seed)\n        return Image.fromarray(result.data[0].copy()), {\n            \'model\': OPENVINO_MODEL, \'device\': f\'OpenVINO {self.device}\',\n            \'load_seconds\': load_time, \'denoising_steps\': steps,\n            \'start_timestep\': 250 * steps - 1, \'inference_seconds\': time.perf_counter() - start,\n        }\n\n\nclass Generator:\n    def __init__(self):\n        self.pipe = None\n        self.loaded_model = None\n        self.openvino = OpenVinoZoom()\n        self.model_id = os.environ.get(\'ZOOM_MODEL\', \'stabilityai/sd-turbo\')\n        if self.model_id not in (\'stabilityai/sd-turbo\', \'stabilityai/sdxl-turbo\'):\n            raise ValueError(\'ZOOM_MODEL must be stabilityai/sd-turbo or stabilityai/sdxl-turbo\')\n\n    def _load_pipeline(self, model_id):\n        if self.loaded_model == model_id:\n            return 0\n        self.pipe = None\n        self.loaded_model = None\n        start = time.perf_counter()\n        import torch\n        from huggingface_hub.constants import HF_HUB_OFFLINE\n\n        # Avoid AutoPipeline: it eagerly imports unrelated Qwen pipelines.\n        if model_id == Z_IMAGE:\n            from diffusers import ZImageImg2ImgPipeline as Pipeline\n        elif model_id == \'stabilityai/sdxl-turbo\':\n            from diffusers import StableDiffusionXLImg2ImgPipeline as Pipeline\n        else:\n            from diffusers import StableDiffusionImg2ImgPipeline as Pipeline\n\n        device = os.environ.get(\'ZOOM_DEVICE\', \'auto\')\n        if device == \'auto\':\n            device = \'cuda\' if torch.cuda.is_available() else (\'mps\' if torch.backends.mps.is_available() else \'cpu\')\n        dtype = torch.float16 if device in (\'cuda\', \'mps\') else torch.float32\n        kwargs = {\'torch_dtype\': dtype}\n        if model_id == Z_IMAGE and device != \'cpu\':\n            kwargs[\'torch_dtype\'] = torch.bfloat16\n        if dtype == torch.float16 and model_id != Z_IMAGE:\n            kwargs[\'variant\'] = \'fp16\'\n        if device == \'cpu\' and \'ZOOM_CPU_THREADS\' in os.environ:\n            torch.set_num_threads(int(os.environ[\'ZOOM_CPU_THREADS\']))\n\n        print(f\'Loading {model_id} on {device} from the local cache...\', flush=True)\n        try:\n            pipe = Pipeline.from_pretrained(model_id, local_files_only=True, **kwargs)\n        except OSError:\n            if HF_HUB_OFFLINE:\n                raise\n            print(f\'Fetching missing model files for {model_id}...\', flush=True)\n            pipe = Pipeline.from_pretrained(model_id, **kwargs)\n        pipe.to(device)\n        if model_id != Z_IMAGE:\n            pipe.enable_attention_slicing()\n        self.pipe = pipe\n        self.loaded_model = model_id\n        return time.perf_counter() - start\n\n    def warmup(self, backend=\'diffusion\'):\n        """Load the default diffusion pipeline before the first click."""\n        if backend == \'openvino\':\n            if self.openvino.pipe is None:\n                start = time.perf_counter()\n                image = Image.new(\'RGB\', (self.openvino.size, self.openvino.size), (120, 125, 130))\n                self.openvino.generate(image, \'Natural detail\', steps=1, seed=42, sharpen=0)\n                return time.perf_counter() - start\n            return 0\n        return self._load_pipeline(self.model_id)\n\n    def generate(self, image, prompt, refinement_steps, seed, backend, sharpen=0):\n        if backend == \'upscale\':\n            return image.copy(), {\'model\': \'Lanczos baseline\', \'load_seconds\': 0, \'inference_seconds\': 0, \'denoising_steps\': 0, \'start_timestep\': None}\n        is_z_image = backend == \'z-image\'\n        model_id = Z_IMAGE if is_z_image else self.model_id\n        schedule_steps = 8 if is_z_image else 4\n        if not 1 <= refinement_steps <= schedule_steps:\n            raise ValueError(f\'Refinement steps must be between 1 and {schedule_steps}\')\n        if backend == \'openvino\':\n            self.pipe = self.loaded_model = None\n            return self.openvino.generate(image, prompt, refinement_steps, seed, sharpen)\n        self.openvino.pipe = None\n        load_time = self._load_pipeline(model_id)\n        import torch\n\n        start = time.perf_counter()\n        input_size = 1024 if is_z_image else 512\n        image = image.resize((input_size, input_size), Image.Resampling.LANCZOS)\n        image = sharpen_input(image, sharpen)\n        used_timesteps = []\n        def record_step(pipe, index, timestep, callback_kwargs):\n            used_timesteps.append(float(timestep))\n            return callback_kwargs\n        # CPU RNG also works with MPS. Each request owns its seed.\n        result = self.pipe(prompt=prompt, image=image, strength=refinement_steps / schedule_steps,\n                           num_inference_steps=schedule_steps, guidance_scale=0.0,\n                           generator=torch.Generator(device=\'cpu\').manual_seed(seed),\n                           callback_on_step_end=record_step)\n        if getattr(result, \'nsfw_content_detected\', None) and any(result.nsfw_content_detected):\n            raise ValueError(\'The model filtered this result; try a different crop or prompt.\')\n        # Release unused temporary Metal allocations before the next local LLM call.\n        if self.pipe.device.type == \'mps\':\n            torch.mps.empty_cache()\n        return result.images[0].convert(\'RGB\'), {\n            \'model\': model_id, \'load_seconds\': load_time,\n            \'denoising_steps\': len(used_timesteps),\n            \'start_timestep\': used_timesteps[0] if used_timesteps else None,\n            \'inference_seconds\': time.perf_counter()-start}\n',
    'imaging.py': '"""Relative zoom geometry and conservative detail compositing."""\nimport base64\nimport io\n\nimport numpy as np\nfrom PIL import Image, ImageChops, ImageFilter\n\n\nFRAME_SIZE = 1024\n\n\ndef decode_zoom_crop(encoded_image, center, zoom_factor=2):\n    """Return crop pixels and the corresponding bounds in 1024px display coordinates."""\n    if len(center) != 2 or not all(0 <= value <= 1 for value in center):\n        raise ValueError(\'Center must lie inside the image\')\n    try:\n        data = base64.b64decode(encoded_image.split(\',\')[-1], validate=True)\n        with Image.open(io.BytesIO(data)) as source:\n            if source.size not in ((FRAME_SIZE, FRAME_SIZE), (2 * FRAME_SIZE, 2 * FRAME_SIZE)):\n                raise ValueError(\'Unsupported source dimensions\')\n            size = FRAME_SIZE // zoom_factor\n            left, top = (\n                max(0, min(FRAME_SIZE - size, int(value * FRAME_SIZE + 0.5) - size // 2))\n                for value in center\n            )\n            bounds = (left, top, left + size, top + size)\n            scale = source.width // FRAME_SIZE\n            return source.crop(tuple(value * scale for value in bounds)).convert(\'RGB\'), bounds\n    except (ValueError, OSError) as exc:\n        raise ValueError(\'Invalid input image; expected a 1024px or 2048px square frame\') from exc\n\n\ndef crop_focus(center, bounds):\n    """Express the original click in the coordinate system of the clamped crop."""\n    left, top, right, bottom = bounds\n    return ((center[0] * FRAME_SIZE - left) / (right - left),\n            (center[1] * FRAME_SIZE - top) / (bottom - top))\n\n\ndef local_fine_contrast(image):\n    lightness = image.convert(\'L\')\n    fine = ImageChops.difference(lightness, lightness.filter(ImageFilter.GaussianBlur(2)))\n    return np.asarray(fine.filter(ImageFilter.GaussianBlur(5)), dtype=np.float32)\n\n\ndef preserve_source_detail(source, generated, focus):\n    """Keep the source\'s broad structure and favor bounded new texture near the click."""\n    source_rgb = np.asarray(source, dtype=np.float32)\n    delta = np.asarray(generated, dtype=np.float32) - source_rgb\n    blur = ImageFilter.GaussianBlur(source.width / 128)\n    broad_delta = (np.asarray(generated.filter(blur), dtype=np.float32)\n                   - np.asarray(source.filter(blur), dtype=np.float32))\n    delta -= broad_delta\n    # A smooth falloff works even when the click is at the very edge of the crop.\n    x = np.linspace(0, 1, source.width, dtype=np.float32)\n    y = np.linspace(0, 1, source.height, dtype=np.float32)[:, None]\n    distance = (x - focus[0]) ** 2 + (y - focus[1]) ** 2\n    weight = np.exp(-distance / (2 * 0.28 ** 2))[..., None]\n    # Limit large changes, which are often a shifted edge or an invented object.\n    np.clip(broad_delta, -42, 42, out=broad_delta)\n    np.clip(delta, -42, 42, out=delta)\n    # Resist replacing real fine detail in areas where the render is smoother.\n    source_contrast = local_fine_contrast(source)\n    generated_contrast = local_fine_contrast(generated)\n    texture_weight = np.clip((generated_contrast + 1) / (source_contrast + 1), 0.25, 1)[..., None]\n    adjustment = (0.10 + 0.22 * weight) * broad_delta + (0.28 + 0.54 * weight) * texture_weight * delta\n    output = np.clip(source_rgb + adjustment, 0, 255)\n    return Image.fromarray(output.round().astype(np.uint8))\n\n\ndef feather_patch(background, patch, feather):\n    """Blend an aligned square patch at its output resolution."""\n    if feather == 0:\n        return patch.copy()\n    size = patch.width\n    axis = np.minimum(np.arange(size), np.arange(size)[::-1])\n    distance = np.minimum(axis[:, None], axis[None, :]).astype(np.float32)\n    mask = (0.5 - 0.5 * np.cos(np.pi * np.clip(distance / feather, 0, 1)))[..., None]\n    blended = mask * np.asarray(patch) + (1 - mask) * np.asarray(background)\n    return Image.fromarray(blended.round().astype(np.uint8))\n\n\ndef sharpen_input(image, amount=0):\n    """Unsharp mask before VAE encoding/noising; amount=0 disables it."""\n    if amount == 0:\n        return image.copy()\n    return image.filter(ImageFilter.UnsharpMask(radius=1.5, percent=round(100*amount), threshold=3))\n',
    'static/index.html': '<!doctype html>\n<html lang="en">\n<head>\n<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">\n<meta name="color-scheme" content="dark">\n<title>Generative Zoom</title>\n<link rel="stylesheet" href="static/style.css">\n</head>\n<body>\n<header class="page-header">\n  <h1>Generative Zoom</h1>\n  <button id="openImage" class="primary" type="button">Open image</button>\n  <input id="upload" hidden type="file" accept="image/*">\n</header>\n<div class="layout">\n  <main aria-label="Image workspace">\n    <div class="view-bar">\n      <div class="navigation">\n        <button id="back" type="button" disabled>Back one zoom</button>\n        <button id="resetView" type="button" disabled>Original image</button>\n      </div>\n      <output id="zoomScale" aria-label="Magnification">1×</output>\n    </div>\n    <div id="stage" class="stage">\n      <canvas id="canvas" width="1024" height="1024" tabindex="0"\n              aria-label="Zoomable image" aria-describedby="canvasHelp"></canvas>\n      <div id="emptyView" class="empty-view">Drop an image here</div>\n      <div id="cropGuide" class="crop-guide" hidden></div>\n      <div id="clickGuide" class="click-guide" hidden></div>\n    </div>\n    <div class="info-bar">\n      <div>\n        <p id="status" role="status">Open or drop an image to start.</p>\n        <p id="timings" class="timings"></p>\n      </div>\n      <label class="compare">View\n        <select id="view" disabled>\n          <option value="image">Blended result</option>\n          <option value="raw">Raw generation</option>\n          <option value="baseline">Lanczos baseline</option>\n        </select>\n      </label>\n    </div>\n    <p id="canvasHelp" class="hint">Tap or click to zoom. <span>With the image focused, use arrow keys to aim and Enter to zoom. Backspace goes back.</span></p>\n  </main>\n  <aside aria-label="Zoom settings">\n    <section class="panel">\n      <h2>Zoom settings</h2>\n      <p id="warmupStatus" class="notice" role="status" hidden></p>\n      <label>Backend\n        <select id="backend">\n          <option value="diffusion">SD-Turbo</option>\n          <option id="openvinoBackend" value="openvino" hidden>SD-Turbo · OpenVINO</option>\n          <option value="z-image">Z-Image-Turbo · 1024×1024</option>\n          <option value="upscale">Lanczos only</option>\n        </select>\n      </label>\n      <div class="paired">\n        <label>Zoom per click <select id="zoomFactor"><option value="2" selected>2×</option><option value="4">4×</option></select></label>\n        <label>Refinement steps <select id="refinementSteps"><option value="1" selected>1 step</option></select></label>\n      </div>\n      <label class="check"><input id="preserveStructure" type="checkbox" checked> Preserve source structure</label>\n    </section>\n    <section class="panel">\n      <h2>Artistic direction</h2>\n      <label>Preset <select id="preset"><option value="">Choose a preset</option><option value="natural">Living ecosystem</option><option value="architecture">Nested architecture</option><option value="surreal">World within a world</option><option value="material">Material microstructure</option></select></label>\n      <label>Prompt hint (optional)<textarea id="prompt" rows="3" maxlength="2000" placeholder="Describe the selected area or suggest a style"></textarea></label>\n      <label>Prompt mode\n        <select id="promptMode">\n          <option value="crop">Crop / manual prompt</option>\n          <option id="backgroundOption" value="background" hidden>Background Qwen</option>\n          <option value="wait" selected>Wait for Qwen</option>\n        </select>\n      </label>\n      <p id="promptStatus" class="hint" role="status" hidden></p>\n      <label id="variationRow">Detail variation <select id="variation"><option value="gentle">Gentle</option><option value="medium" selected>Medium</option><option value="surprising">Surprising</option></select></label>\n      <details id="promptDetails" hidden>\n        <summary id="promptSummary">Prompt used for the last image</summary>\n        <p id="detailHeading" class="detail-heading"></p>\n        <p id="detailDirection"></p>\n        <p id="cropObservation"></p>\n        <p id="usedPrompt"></p>\n      </details>\n    </section>\n    <details class="panel advanced">\n      <summary>Advanced settings</summary>\n      <label>Pre-noise sharpening <input id="sharpen" type="number" min="0" max="3" step="any" value="0" required></label>\n      <label>Border feather (px at 512px) <input id="feather" type="number" min="0" max="128" value="32" required></label>\n      <label>Seed <input id="seed" type="number" min="0" max="2147483647" value="42" required></label>\n    </details>\n  </aside>\n</div>\n<script type="module" src="static/app.js"></script>\n</body></html>\n',
    'static/app.js': "import {maxExtraDepth, promptApplies} from './prompting.mjs';\n\nconst $ = id => document.getElementById(id);\nconst canvas = $('canvas'), ctx = canvas.getContext('2d');\nconst stage = $('stage');\nconst zoomHistory = [];\nconst historyLimit = 6;\nconst maxEncodedFrameLength = 8_000_000;\nconst centerPoint = [0.5, 0.5];\nlet frame = null, original = null, focus = centerPoint, showGuide = false, busy = false;\nlet warmup = {state: 'idle'}, backendChosen = false;\nlet nextFrameId = 0;\nlet promptModeChosen = false;\nlet aimTimer;\nconst background = {revision: 0, pendingRevision: null, ready: null, error: '', retryAt: 0};\n\nconst clamp = (x, a, b) => Math.max(a, Math.min(b, x));\nfunction cropBounds(point, zoomFactor) {\n  const size = canvas.width;\n  const width = size / zoomFactor;\n  const [left, top] = point.map(value => clamp(Math.round(value * size) - width / 2, 0, size - width));\n  return [left, top, left + width, top + width];\n}\n\nfunction updateGuide() {\n  const visible = Boolean(frame && showGuide && !busy);\n  $('cropGuide').hidden = $('clickGuide').hidden = !visible;\n  if (!visible) return;\n  const [left, top, right] = cropBounds(focus, Number($('zoomFactor').value));\n  const percent = value => `${100 * value / canvas.width}%`;\n  $('cropGuide').style.left = percent(left);\n  $('cropGuide').style.top = percent(top);\n  $('cropGuide').style.width = percent(right - left);\n  $('clickGuide').style.left = `${focus[0] * 100}%`;\n  $('clickGuide').style.top = `${focus[1] * 100}%`;\n}\n\nfunction draw() {\n  ctx.clearRect(0, 0, canvas.width, canvas.height);\n  const image = frame?.decodedViews?.[$('view').value] || frame?.image;\n  if (image) ctx.drawImage(image, 0, 0, canvas.width, canvas.height);\n  updateGuide();\n}\n\nfunction showDetail(detail) {\n  $('promptDetails').hidden = !detail;\n  $('promptSummary').textContent = detail?.summary || 'Prompt used for the last image';\n  $('detailHeading').textContent = detail?.heading || '';\n  $('detailDirection').textContent = detail?.direction || '';\n  $('cropObservation').textContent = detail?.observation || '';\n  $('usedPrompt').textContent = detail?.prompt || '';\n}\n\nfunction waitingForModel() {\n  return warmup.state === 'warming' && $('backend').value !== 'upscale';\n}\n\nfunction promptMode() {\n  return $('backend').value === 'upscale' ? 'crop' : $('promptMode').value;\n}\n\nfunction promptTarget(source) {\n  const zoomFactor = Number($('zoomFactor').value);\n  return {\n    frameId: source.id, depth: source.depth, trail: source.trail, center: focus, zoomFactor,\n    bounds: cropBounds(focus, zoomFactor).map(value => value / canvas.width),\n    settings: JSON.stringify([$('prompt').value.trim(), $('variation').value, Number($('seed').value)])\n  };\n}\n\nfunction directorBody(source, target) {\n  const [hint, variation, seed] = JSON.parse(target.settings);\n  return {\n    image: source.referenceImage || source.image.src, center: target.center, zoom_factor: target.zoomFactor,\n    ...(hint ? {initial_prompt: hint} : {}), step: source.detailStep + 1,\n    zoom_depth: source.depth, variation, seed\n  };\n}\n\nfunction cachedPrompt(target) {\n  const entry = background.ready;\n  return entry && promptApplies(entry.target, target) ? entry.detail : null;\n}\n\nfunction updatePromptStatus() {\n  const status = $('promptStatus');\n  status.hidden = promptMode() !== 'background' || !frame;\n  if (status.hidden) return;\n  let message = 'Aim at a detail or click to start preparing a prompt.';\n  const ready = cachedPrompt(promptTarget(frame));\n  if (ready) message = `Qwen prompt ready for this area · ${ready.seconds.toFixed(1)}s analysis.`;\n  else if (background.pendingRevision === background.revision) message = 'Qwen is preparing a prompt. You can keep zooming.';\n  else if (background.pendingRevision !== null) message = 'An earlier Qwen request is finishing. You can keep zooming.';\n  else if (Date.now() < background.retryAt) message = background.error;\n  if (status.textContent !== message) status.textContent = message;\n}\n\nfunction clearBackgroundPrompt() {\n  clearTimeout(aimTimer);\n  background.revision++;\n  background.ready = null;\n  background.error = '';\n  background.retryAt = 0;\n  updatePromptStatus();\n}\n\nasync function preparePrompt(source, target) {\n  if (background.pendingRevision !== null || Date.now() < background.retryAt || cachedPrompt(target)) return;\n  const revision = background.revision;\n  background.pendingRevision = revision;\n  updatePromptStatus();\n  try {\n    const detail = await requestDetail(directorBody(source, target));\n    if (revision === background.revision) background.ready = {target, detail};\n  } catch (error) {\n    if (revision === background.revision) {\n      background.retryAt = Date.now() + (error.status === 409 ? 5000 : 30000);\n      background.error = `Background prompt unavailable. ${error.message}`;\n    }\n  } finally {\n    background.pendingRevision = null;\n    updatePromptStatus();\n  }\n}\n\nfunction schedulePrompt() {\n  clearTimeout(aimTimer);\n  updatePromptStatus();\n  if (!frame || promptMode() !== 'background' || waitingForModel() || !$('seed').checkValidity()) return;\n  aimTimer = setTimeout(() => {\n    if (!busy && showGuide) void preparePrompt(frame, promptTarget(frame));\n  }, 650);\n}\n\nfunction updateControls() {\n  for (const element of document.querySelectorAll('input, textarea, select, button')) {\n    element.disabled = busy;\n  }\n  $('back').disabled = busy || !zoomHistory.length;\n  $('resetView').disabled = busy || frame === original;\n  $('view').disabled = busy || !frame?.views;\n  const lanczos = $('backend').value === 'upscale';\n  $('refinementSteps').disabled = $('preserveStructure').disabled = busy || lanczos;\n  $('preset').disabled = $('prompt').disabled = busy || lanczos || !frame;\n  $('promptMode').disabled = busy || lanczos;\n  $('variationRow').hidden = promptMode() === 'crop';\n  $('warmupStatus').hidden = warmup.state !== 'warming' && warmup.state !== 'failed';\n  if (warmup.state === 'warming') {\n    $('warmupStatus').textContent = `${warmup.message} You can open an image or use Lanczos.`;\n  } else if (warmup.state === 'failed') {\n    $('warmupStatus').textContent = `Warm-up did not finish. ${warmup.message}`;\n  }\n  stage.toggleAttribute('data-image', Boolean(frame));\n  stage.toggleAttribute('data-busy', busy || waitingForModel());\n  canvas.setAttribute('aria-busy', String(busy));\n  $('emptyView').hidden = Boolean(frame);\n  updateGuide();\n  updatePromptStatus();\n}\n\nfunction showFrame(next, point = centerPoint) {\n  clearTimeout(aimTimer);\n  frame = next;\n  focus = point;\n  $('view').value = 'image';\n  $('zoomScale').textContent = frame.depth < 20 ? `${2 ** frame.depth}×` : `2^${frame.depth}×`;\n  $('status').textContent = frame.status;\n  $('timings').textContent = frame.timings || '';\n  showDetail(frame.detail);\n  updateControls();\n  draw();\n}\n\nfunction loadImage(url) {\n  return new Promise((resolve, reject) => {\n    const image = new Image();\n    image.onload = () => resolve(image);\n    image.onerror = () => reject(new Error('Could not open this image. Try a PNG, JPEG, or WebP.'));\n    image.src = url;\n  });\n}\n\nfunction squarePreview(source, resolution) {\n  const size = Math.min(source.naturalWidth, source.naturalHeight);\n  const square = document.createElement('canvas');\n  square.width = square.height = resolution;\n  const squareContext = square.getContext('2d', {alpha: false});\n  squareContext.fillStyle = '#191d20';\n  squareContext.fillRect(0, 0, resolution, resolution);\n  squareContext.drawImage(source, (source.naturalWidth - size) / 2, (source.naturalHeight - size) / 2,\n    size, size, 0, 0, resolution, resolution);\n  return square;\n}\n\nasync function openImage(file) {\n  if (!file || busy) return;\n  const url = URL.createObjectURL(file);\n  busy = true;\n  updateControls();\n  $('status').textContent = 'Opening image…';\n  try {\n    const source = await loadImage(url);\n    const size = Math.min(source.naturalWidth, source.naturalHeight);\n    const reference = size >= 1.5 * canvas.width\n      ? squarePreview(source, 2 * canvas.width).toDataURL('image/jpeg', 0.96) : null;\n    const referenceImage = reference?.length <= maxEncodedFrameLength ? reference : null;\n    const image = await loadImage(squarePreview(source, canvas.width).toDataURL());\n    original = {image, referenceImage, id: ++nextFrameId, trail: [], depth: 0, detailStep: 0,\n      status: 'Choose a point to zoom in.'};\n    zoomHistory.length = 0;\n    $('preset').value = $('prompt').value = '';\n    clearBackgroundPrompt();\n    showFrame(original);\n  } catch (error) {\n    $('status').textContent = error.message;\n  } finally {\n    URL.revokeObjectURL(url);\n    busy = false;\n    updateControls();\n  }\n}\n\n$('openImage').onclick = () => $('upload').click();\n$('upload').onchange = event => {\n  const file = event.target.files[0];\n  event.target.value = '';\n  openImage(file);\n};\nstage.ondragover = event => {\n  if (!event.dataTransfer.types.includes('Files')) return;\n  event.preventDefault();\n  stage.classList.toggle('dragging', !busy);\n};\nstage.ondragleave = () => stage.classList.remove('dragging');\nstage.ondrop = event => {\n  event.preventDefault();\n  stage.classList.remove('dragging');\n  openImage(event.dataTransfer.files[0]);\n};\n\nfunction pointerFocus(event) {\n  const rect = canvas.getBoundingClientRect();\n  return [clamp((event.clientX - rect.left) / rect.width, 0, 1),\n    clamp((event.clientY - rect.top) / rect.height, 0, 1)];\n}\n\ncanvas.onpointermove = event => {\n  if (busy || event.pointerType === 'touch') return;\n  focus = pointerFocus(event);\n  showGuide = true;\n  updateGuide();\n  schedulePrompt();\n};\ncanvas.onpointerleave = canvas.onblur = () => {\n  showGuide = false;\n  clearTimeout(aimTimer);\n  updateGuide();\n};\ncanvas.onfocus = () => {\n  showGuide = true;\n  updateGuide();\n};\ncanvas.onpointerup = event => {\n  if (busy || !frame || !event.isPrimary || event.button !== 0) return;\n  focus = pointerFocus(event);\n  generate();\n};\ncanvas.onkeydown = event => {\n  if (busy || !frame || event.ctrlKey || event.metaKey || event.altKey) return;\n  const movement = {ArrowLeft: [-1, 0], ArrowRight: [1, 0], ArrowUp: [0, -1], ArrowDown: [0, 1]}[event.key];\n  if (movement) {\n    event.preventDefault();\n    const amount = event.shiftKey ? 0.01 : 0.04;\n    focus = focus.map((value, axis) => clamp(value + movement[axis] * amount, 0, 1));\n    showGuide = true;\n    updateGuide();\n    schedulePrompt();\n  } else if (event.key === 'Enter' || event.key === ' ') {\n    event.preventDefault();\n    if (!event.repeat) generate();\n  } else if (event.key === 'Backspace') {\n    event.preventDefault();\n    if (!event.repeat) goBack();\n  }\n};\nfunction goBack() {\n  if (busy || !zoomHistory.length) return;\n  const step = zoomHistory.pop();\n  $('zoomFactor').value = String(step.zoomFactor);\n  showFrame(step.frame, step.center);\n  canvas.focus({preventScroll: true});\n}\n$('back').onclick = goBack;\n$('resetView').onclick = () => {\n  if (busy || !original) return;\n  zoomHistory.length = 0;\n  clearBackgroundPrompt();\n  showFrame(original);\n};\n$('view').onchange = async () => {\n  const selected = $('view').value;\n  const target = frame;\n  const encoded = target?.views?.[selected];\n  try {\n    if (encoded && !target.decodedViews[selected]) {\n      target.decodedViews[selected] = await loadImage(encoded);\n    }\n    if (frame === target && $('view').value === selected && !busy) draw();\n  } catch (error) {\n    if (frame !== target || $('view').value !== selected || busy) return;\n    $('view').value = 'image';\n    $('status').textContent = error.message;\n    draw();\n  }\n};\n$('zoomFactor').onchange = () => {\n  clearTimeout(aimTimer);\n  updateGuide();\n  updatePromptStatus();\n};\n\nfunction previewZoom(source, bounds) {\n  const reducedMotion = window.matchMedia('(prefers-reduced-motion: reduce)').matches;\n  const size = canvas.width;\n  const [previewLeft, previewTop, previewRight] = bounds;\n  const cropSize = previewRight - previewLeft;\n  const previewDuration = 260;\n  const minimumRevealDuration = 160;\n  const extraTravelDuration = 150;\n  let frame;\n  let progress = 0;\n  let previewStart;\n  const ease = t => t * t * (3 - 2 * t);\n\n  function render(left, top, width, image, opacity = 0) {\n    const span = size + (width - size) * progress;\n    const viewLeft = left * progress;\n    const viewTop = top * progress;\n    ctx.drawImage(source, viewLeft, viewTop, span, span, 0, 0, size, size);\n    if (image) {\n      const scale = size / span;\n      ctx.globalAlpha = opacity;\n      ctx.drawImage(image, (left - viewLeft) * scale, (top - viewTop) * scale,\n        width * scale, width * scale);\n      ctx.globalAlpha = 1;\n    }\n  }\n\n  function preview(now) {\n    previewStart ??= now;\n    const elapsed = clamp((now - previewStart) / previewDuration, 0, 1);\n    // Fill the view with the exact crop immediately while the new detail renders.\n    progress = 1 - (1 - elapsed) ** 3;\n    render(previewLeft, previewTop, cropSize);\n    if (elapsed < 1) frame = requestAnimationFrame(preview);\n  }\n\n  if (reducedMotion) {\n    progress = 1;\n    render(previewLeft, previewTop, cropSize);\n  } else {\n    frame = requestAnimationFrame(preview);\n  }\n  const cancel = () => cancelAnimationFrame(frame);\n  return {\n    cancel,\n    finish(image, [left, top, right]) {\n      if (reducedMotion) return;\n      cancel();\n      const from = progress;\n      const duration = minimumRevealDuration + extraTravelDuration * (1 - from);\n      let revealStart;\n      return new Promise(resolve => {\n        function reveal(now) {\n          revealStart ??= now;\n          const t = clamp((now - revealStart) / duration, 0, 1);\n          const blend = ease(t);\n          progress = from + (1 - from) * blend;\n          render(left, top, right - left, image, blend);\n          if (t < 1) frame = requestAnimationFrame(reveal);\n          else resolve();\n        }\n        frame = requestAnimationFrame(reveal);\n      });\n    }\n  };\n}\nasync function requestDetail(body, trackStatus = false) {\n  const start = performance.now();\n  const controller = new AbortController();\n  let timer;\n  async function showStage() {\n    try {\n      const response = await fetch('api/director/status', {cache: 'no-store', signal: controller.signal});\n      const {stage} = await response.json();\n      if (stage && !controller.signal.aborted) {\n        const seconds = Math.floor((performance.now() - start) / 1000);\n        $('status').textContent = `${stage} · ${seconds}s elapsed`;\n      }\n    } catch {\n      // The main request reports generation errors even when status polling fails.\n    } finally {\n      if (!controller.signal.aborted) timer = setTimeout(showStage, 3000);\n    }\n  }\n  if (trackStatus) timer = setTimeout(showStage, 1000);\n  try {\n    const response = await fetch('api/director', {\n      method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(body)\n    });\n    const detail = await response.json();\n    if (!response.ok) {\n      const error = new Error(typeof detail.detail === 'string' ? detail.detail : JSON.stringify(detail.detail));\n      error.status = response.status;\n      throw error;\n    }\n    return detail;\n  } finally {\n    controller.abort();\n    clearTimeout(timer);\n  }\n}\nasync function generate() {\n  if (!frame || busy) return;\n  if (waitingForModel()) {\n    $('status').textContent = 'The model is still preparing. You can use Lanczos in the meantime.';\n    return;\n  }\n  for (const id of ['seed', 'sharpen', 'feather']) {\n    if (!$(id).checkValidity()) {\n      $('status').textContent = 'Check the seed, sharpening, and feather values in Advanced settings.';\n      return;\n    }\n  }\n  const started = performance.now();\n  const previous = frame;\n  const mode = promptMode();\n  const target = promptTarget(previous);\n  let nextDetail = mode === 'background' ? cachedPrompt(target) : null;\n  let promptWarning = '';\n  const payload = {\n    image: previous.referenceImage || previous.image.src,\n    center: focus,\n    zoom_factor: Number($('zoomFactor').value),\n    prompt: $('prompt').value.trim(),\n    backend: $('backend').value,\n    refinement_steps: Number($('refinementSteps').value),\n    seed: Number($('seed').value),\n    sharpen: Number($('sharpen').value),\n    feather: Number($('feather').value),\n    preserve_structure: $('preserveStructure').checked\n  };\n  const promptSource = payload.prompt ? 'Manual prompt' : 'Image crop only';\n  const fallbackSource = payload.prompt ? 'your prompt hint' : 'the image crop';\n  clearTimeout(aimTimer);\n  busy = true;\n  $('view').value = 'image';\n  $('timings').textContent = '';\n  updateControls();\n  const zoom = previewZoom(previous.image, cropBounds(payload.center, payload.zoom_factor));\n  try {\n    if (mode === 'wait') {\n      $('status').textContent = 'Reading the selected crop and writing a detail direction…';\n      try {\n        nextDetail = await requestDetail(directorBody(previous, target), true);\n        payload.prompt = nextDetail.prompt;\n      } catch (error) {\n        promptWarning = error instanceof TypeError ? 'The prompt request lost its connection.' : error.message;\n      }\n      showDetail({\n        summary: 'Prompt for this zoom attempt',\n        heading: nextDetail ? 'Detail direction for this zoom' : promptSource,\n        direction: nextDetail?.detail || promptWarning,\n        observation: nextDetail ? `Visible in the analyzed crop: ${nextDetail.observations}` : '',\n        prompt: payload.prompt\n      });\n    }\n    if (mode === 'background' && nextDetail) payload.prompt = nextDetail.prompt;\n    $('status').textContent = promptWarning\n      ? `Qwen unavailable. Generating with ${fallbackSource}…`\n      : 'Generating the image…';\n    const rendering = fetch('api/zoom', {\n      method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(payload)\n    });\n    if (mode === 'background' && !nextDetail) void preparePrompt(previous, target);\n    const response = await rendering;\n    const data = await response.json();\n    if (!response.ok) throw new Error(typeof data.detail === 'string' ? data.detail : JSON.stringify(data.detail));\n    const image = await loadImage(data.image);\n    await zoom.finish(image, data.crop);\n    zoomHistory.push({frame: previous, center: target.center, zoomFactor: target.zoomFactor});\n    if (zoomHistory.length > historyLimit) zoomHistory.shift();\n    if (warmup.state === 'failed' && payload.backend !== 'upscale') warmup = {state: 'ready'};\n    const detailStep = previous.detailStep + Number(Boolean(nextDetail));\n    const lanczos = payload.backend === 'upscale';\n    const timingParts = [];\n    if (data.device) timingParts.push(data.device);\n    if (!lanczos) {\n      timingParts.push(`Inference ${data.inference_seconds.toFixed(2)}s`);\n      if (data.load_seconds > 0) timingParts.push(`Model load ${data.load_seconds.toFixed(2)}s`);\n      if (nextDetail) timingParts.push(mode === 'background' ? 'Background Qwen used' : `Qwen ${nextDetail.seconds.toFixed(2)}s`);\n    }\n    const totalTime = ((performance.now() - started) / 1000).toFixed(2);\n    timingParts.push(`Click to result ${totalTime}s`);\n    const nextFocus = target.center.map((value, axis) =>\n      (Math.round(value * canvas.width) - data.crop[axis]) / (data.crop[2] - data.crop[0]));\n    showFrame({\n      image,\n      id: ++nextFrameId,\n      trail: [{frameId: target.frameId, center: target.center, zoomFactor: target.zoomFactor, bounds: target.bounds},\n        ...previous.trail].slice(0, maxExtraDepth),\n      views: lanczos ? null : {raw: data.raw, baseline: data.baseline},\n      decodedViews: {},\n      depth: previous.depth + Math.log2(payload.zoom_factor),\n      detailStep,\n      detail: lanczos ? null : {\n        heading: nextDetail ? (mode === 'background' ? 'Background Qwen prompt' : `Detail direction · Step ${detailStep}`) : promptSource,\n        direction: nextDetail?.detail || promptWarning,\n        observation: nextDetail ? `Visible in the analyzed crop: ${nextDetail.observations}` : '',\n        prompt: payload.prompt\n      },\n      status: lanczos ? 'Lanczos zoom complete.' : promptWarning\n        ? `Zoom complete using ${fallbackSource}.` : 'Generated zoom complete.',\n      timings: timingParts.join(' · ')\n    }, nextFocus);\n  } catch(e) {\n    $('status').textContent = e instanceof TypeError\n      ? 'The request to the zoom server failed. Check the Python terminal for the error.'\n      : e.message;\n  }\n  finally {\n    zoom.cancel();\n    busy = false;\n    updateControls();\n    draw();\n  }\n}\nconst prompts = {\n  natural: 'A close-up photograph of an intricate living ecosystem, interwoven roots, branching moss, tiny fern fronds, mineral grains and glistening water droplets, irregular organic structures at many scales, crisp fine detail throughout, natural diffuse light',\n  architecture: 'An intricate architectural world of nested courtyards, narrow staircases, carved stone arches, small balconies and connected passages, structures within structures, weathered materials, coherent perspective, crisp detail throughout, soft daylight',\n  surreal: 'A miniature world within an organic landscape, branching river channels surrounding tiny terraced gardens, intricate structures nested inside larger forms, coherent natural materials and illumination, richly varied small-scale detail, crisp focus throughout',\n  material: 'An extreme close-up of layered natural material, branching fissures, interlocking crystalline grains, tiny pores and translucent mineral veins, irregular multiscale structure, crisp surface detail throughout, diffuse lighting'\n};\n$('preset').onchange = () => {\n  $('prompt').value = prompts[$('preset').value] || '';\n  clearBackgroundPrompt();\n};\n$('prompt').oninput = () => {\n  $('preset').value = '';\n  clearBackgroundPrompt();\n};\n$('variation').onchange = $('seed').oninput = clearBackgroundPrompt;\n$('promptMode').onchange = () => {\n  promptModeChosen = true;\n  clearBackgroundPrompt();\n  updateControls();\n};\n\nfunction configureBackend() {\n  const steps = $('refinementSteps');\n  const maxSteps = $('backend').value === 'z-image' ? 8 : 4;\n  const selected = Math.min(Number(steps.value), maxSteps);\n  steps.replaceChildren(...Array.from({length: maxSteps}, (_, index) => {\n    const count = index + 1;\n    return new Option(count === 1 ? '1 step' : count + ' steps', count);\n  }));\n  steps.value = selected;\n  updateControls();\n}\n\n$('backend').onchange = () => {\n  backendChosen = true;\n  clearTimeout(aimTimer);\n  configureBackend();\n};\n\nasync function refreshConfiguration(initial = false) {\n  try {\n    const response = await fetch('api/config', {cache: 'no-store'});\n    if (!response.ok) throw new Error('Configuration request failed');\n    const config = await response.json();\n    const wasWaiting = waitingForModel();\n    warmup = config.warmup || {state: 'idle'};\n    $('openvinoBackend').hidden = !config.openvino_available;\n    $('backgroundOption').hidden = !config.background_prompting;\n    if (initial && config.background_prompting && !promptModeChosen) $('promptMode').value = 'crop';\n    if (initial && config.realtime && config.openvino_available && !backendChosen) {\n      $('backend').value = 'openvino';\n      configureBackend();\n    } else {\n      updateControls();\n    }\n    if (wasWaiting && warmup.state === 'ready' && frame && !busy) {\n      $('status').textContent = 'Model ready. Choose a point to zoom in.';\n    }\n  } catch {\n    // If polling was already active, retry a transient disconnection.\n  }\n  if (warmup.state === 'warming') setTimeout(refreshConfiguration, 750);\n}\n\nconfigureBackend();\nrefreshConfiguration(true);\n",
    'static/prompting.mjs': '// Reuse a visual prompt for at most two additional 2× doublings.\nexport const maxExtraDepth = 2;\n\nconst nearby = (point, center, width, fraction) =>\n  point.every((value, axis) => Math.abs(value - center[axis]) <= width * fraction);\n\nconst cropWidth = ({bounds}) => bounds[2] - bounds[0];\n\nfunction sameFocus(first, second) {\n  return first.zoomFactor === second.zoomFactor && nearby(second.center, first.center, cropWidth(first), 0.10);\n}\n\nexport function promptApplies(analysis, target) {\n  if (analysis.settings !== target.settings) return false;\n  const extraDepth = target.depth + Math.log2(target.zoomFactor)\n    - analysis.depth - Math.log2(analysis.zoomFactor);\n  if (extraDepth < 0 || extraDepth > maxExtraDepth) return false;\n  if (analysis.frameId === target.frameId) return sameFocus(analysis, target);\n\n  let point = target.center;\n  for (const step of target.trail) {\n    const [left, top] = step.bounds;\n    const width = cropWidth(step);\n    point = [left + point[0] * width, top + point[1] * width];\n    if (step.frameId === analysis.frameId) {\n      return sameFocus(analysis, step) && nearby(point, analysis.center, cropWidth(analysis), 0.25);\n    }\n  }\n  return false;\n}\n',
    'static/style.css': ':root {\n  color-scheme: dark;\n  font: 15px/1.5 system-ui, sans-serif;\n  --background: #101214;\n  --surface: #191c1e;\n  --raised: #24282a;\n  --border: #373c3f;\n  --text: #f0f1ee;\n  --muted: #aab4b0;\n  --accent: #bce68d;\n}\n\n*, *::before, *::after { box-sizing: border-box; }\n[hidden] { display: none !important; }\nbody { margin: 0 auto; max-width: 1200px; padding: 26px clamp(16px, 4vw, 44px); background: var(--background); color: var(--text); }\nh1, h2, p { margin: 0; }\nh1 { font-size: clamp(23px, 4vw, 32px); font-weight: 650; letter-spacing: -.045em; }\nh2 { font-size: 16px; font-weight: 640; }\nbutton, select, input, textarea { font: inherit; color: inherit; }\nbutton, select, input[type="number"], textarea {\n  border: 1px solid var(--border);\n  border-radius: 7px;\n  background: var(--raised);\n}\nbutton { padding: 7px 11px; cursor: pointer; }\nbutton:hover:enabled { border-color: var(--muted); background: #303537; }\nbutton:disabled, select:disabled { opacity: .5; cursor: default; }\nbutton.primary { background: var(--accent); color: #17200d; border-color: transparent; font-weight: 680; padding-inline: 15px; }\nbutton.primary:hover:enabled { background: #d0f4a8; }\n:focus-visible { outline: 2px solid var(--accent); outline-offset: 3px; }\n\n.page-header, .view-bar, .info-bar { display: flex; align-items: center; justify-content: space-between; gap: 12px; }\n.page-header { border-bottom: 1px solid var(--border); padding-bottom: 18px; margin-bottom: 20px; }\n.layout {\n  display: grid;\n  grid-template-columns: minmax(0, 1fr) minmax(280px, 320px);\n  align-items: start;\n  gap: 24px;\n  width: min(100%, calc(max(480px, 100svh - 225px) + 344px));\n  margin-inline: auto;\n}\n.navigation { display: flex; flex-wrap: wrap; gap: 8px; }\n.view-bar { margin-bottom: 10px; }\n#zoomScale { color: var(--accent); font-variant-numeric: tabular-nums; padding-inline: 4px; }\n.stage { position: relative; aspect-ratio: 1; background: #191d20; border: 1px solid var(--border); border-radius: 10px; overflow: hidden; }\n.stage:focus-within, .stage.dragging { border-color: var(--accent); }\n.stage[data-busy] { cursor: progress; }\ncanvas { display: block; width: 100%; height: 100%; touch-action: manipulation; }\n.stage[data-image]:not([data-busy]) canvas { cursor: crosshair; }\ncanvas:focus-visible { outline-offset: -3px; }\n.empty-view { position: absolute; inset: 0; display: grid; place-items: center; color: var(--muted); pointer-events: none; }\n.crop-guide, .click-guide { position: absolute; pointer-events: none; }\n.crop-guide { aspect-ratio: 1; border: 1px solid var(--accent); box-shadow: 0 0 0 1px #0b100c80; border-radius: 2px; }\n.click-guide { width: 8px; height: 8px; border: 1px solid #1c2914; background: var(--accent); border-radius: 50%; transform: translate(-50%, -50%); }\n.info-bar { align-items: start; margin-top: 12px; }\n#status { white-space: pre-wrap; overflow-wrap: anywhere; }\n.timings, .hint, .compare { color: var(--muted); font-size: 13px; }\n.timings:empty { display: none; }\n.timings { margin-top: 4px; font-variant-numeric: tabular-nums; }\n.hint { margin-top: 10px; }\n.compare { display: flex; align-items: center; gap: 7px; white-space: nowrap; }\n.compare select { min-width: 140px; }\naside { display: grid; gap: 12px; }\n.panel { padding: 15px; border: 1px solid var(--border); border-radius: 10px; background: var(--surface); }\n.panel label { display: block; margin-top: 11px; }\n.paired { display: grid; grid-template-columns: 1fr 1fr; gap: 10px; }\nselect, textarea { padding: 7px 9px; }\n.panel select, .panel textarea { display: block; width: 100%; margin-top: 4px; }\ntextarea { resize: vertical; }\n.panel label.check { display: flex; align-items: center; gap: 8px; }\ninput[type="checkbox"] { accent-color: var(--accent); margin: 0; }\n.notice { margin-top: 8px; color: var(--accent); font-size: 13px; overflow-wrap: anywhere; }\n#promptDetails { border-top: 1px solid var(--border); margin-top: 13px; padding-top: 10px; font-size: 13px; }\n#promptDetails p { margin-top: 6px; overflow-wrap: anywhere; }\n#promptDetails p:empty { display: none; }\n.detail-heading { color: var(--accent); }\nsummary { cursor: pointer; }\n.advanced label { display: flex; align-items: center; justify-content: space-between; gap: 10px; }\ninput[type="number"] { width: 104px; padding: 6px 8px; }\n\n@media (max-width: 790px) {\n  .layout { grid-template-columns: 1fr; width: 100%; }\n  .view-bar { flex-wrap: wrap; }\n}\n\n@media (pointer: coarse) {\n  .hint span { display: none; }\n}\n\n@media (max-width: 420px) {\n  .info-bar { flex-direction: column; gap: 7px; }\n  .navigation button { font-size: 13px; }\n}\n',
    'requirements.txt': 'pydantic>=2,<3\npillow>=10.4,<12\nnumpy>=1.26,<3\n',
    'requirements-model.txt': '-r requirements.txt\ntorch>=2.4\ntransformers>=4.44,<5\ndiffusers>=0.30,<1\naccelerate>=0.34\nsafetensors>=0.4\n',
    'requirements-colab.txt': '-r requirements-model.txt\ntransformers>=4.57,<5\ndiffusers>=0.36,<1\nbitsandbytes>=0.46,<1\n',
}
for name, source in files.items():
    path = project / name
    path.parent.mkdir(exist_ok=True)
    path.write_text(source, encoding='utf-8')


In [ ]:
#@title Check the GPU and install dependencies
import subprocess
import sys
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Choose Runtime → Change runtime type → GPU, reconnect, and run all cells.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
               cwd=project, check=True)

In [ ]:
#@title Start the demo
import os
import time
from urllib.error import URLError
from urllib.request import urlopen
from google.colab import output

if "server" in globals() and server.poll() is None:
    server.terminate()
    server.wait(timeout=15)
log = project / "server.log"
env = dict(os.environ, ZOOM_DEVICE="cuda", ZOOM_PROMPT_BACKEND="transformers",
           ZOOM_PROMPT_MODEL="Qwen/Qwen3-VL-4B-Instruct", ZOOM_MODEL="stabilityai/sd-turbo")
with log.open("w") as stream:
    server = subprocess.Popen([sys.executable, "-u", "app.py"], cwd=project, env=env,
                              stdout=stream, stderr=subprocess.STDOUT)
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError(log.read_text())
    try:
        with urlopen("http://127.0.0.1:8000/static/app.js", timeout=1) as response:
            if response.status == 200:
                break
    except URLError:
        pass
    time.sleep(0.5)
else:
    server.terminate()
    raise RuntimeError("Server startup timed out. " + log.read_text())
output.serve_kernel_port_as_iframe(8000, height=900)

The first evolving zoom loads Qwen and the selected image model. Later zooms reuse both models. **Lanczos only** exercises the upload and click workflow immediately. To inspect a failed request, run `print(log.read_text()[-4000:])` in a new cell. Rerun **Start the demo** to release the models and restart the server.